In [ ]:
"""
═══════════════════════════════════════════════════════════════════
VNDHR Preprocessing Pipeline
Based on: "VNDHR: Variational Single Nighttime Image Dehazing for 
Enhancing Visibility in Intelligent Transportation Systems via 
Hybrid Regularization" (IEEE TITS 2025)

This code preprocesses nighttime hazy images using the VNDHR method
described in the paper. It processes both training and testing datasets.
═══════════════════════════════════════════════════════════════════
"""

import os
import glob
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
from scipy import sparse
from scipy.sparse.linalg import cg, LinearOperator

# ═══════════════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════════════

# Input directories (raw hazy images)
TRAIN_INPUT = r"D:\Downloads\data\train\input"
TEST_INPUT = r"D:\Downloads\data\test\input"

# Output directories (VNDHR preprocessed)
TRAIN_VNDHR = r"D:\Downloads\data\train_vndhr"
TEST_VNDHR = r"D:\Downloads\data\test_vndhr"

# Create output directories
for directory in [TRAIN_VNDHR, TEST_VNDHR]:
    os.makedirs(directory, exist_ok=True)

# VNDHR Parameters (from paper Section IV-A)
VNDHR_PARAMS = {
    'lambda1': 0.002,    # ℓp norm weight for illumination
    'lambda2': 0.0001,   # Weighted ℓ2 norm for reflectance
    'lambda3': 0.001,    # ℓ1 TV norm for noise suppression
    'p': 0.65,           # Fractional-order norm parameter
    'max_iter': 10,      # Maximum iterations
    'tau1': 1e-6,        # Small constant for gradient magnitude
    'tau2': 1e-6,        # Small constant for gradient magnitude
    'epsilon': 0.001     # Convergence threshold
}

IMG_SIZE = 256  # Resize for computational efficiency

print("═"*80)
print("VNDHR PREPROCESSING PIPELINE")
print("Based on IEEE TITS 2025 Paper")
print("═"*80)
print(f"Parameters: λ1={VNDHR_PARAMS['lambda1']}, λ2={VNDHR_PARAMS['lambda2']}, "
      f"λ3={VNDHR_PARAMS['lambda3']}, p={VNDHR_PARAMS['p']}")
print("═"*80 + "\n")

# ═══════════════════════════════════════════════════════════════════
# GRADIENT OPERATORS (from paper Section III-B)
# ═══════════════════════════════════════════════════════════════════

def gradient_operators(H, W):
    """
    Construct discrete gradient operators using Toeplitz matrices
    Returns: Dx, Dy (sparse matrices for horizontal and vertical gradients)
    """
    N = H * W
    
    # Horizontal gradient (Dx)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if c < W - 1:
                right = r * W + (c + 1)
                rowi.extend([idx, idx])
                coli.extend([idx, right])
                di.extend([-1, 1])
    Dx = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    # Vertical gradient (Dy)
    rowi, coli, di = [], [], []
    for r in range(H):
        for c in range(W):
            idx = r * W + c
            if r < H - 1:
                down = (r + 1) * W + c
                rowi.extend([idx, idx])
                coli.extend([idx, down])
                di.extend([-1, 1])
    Dy = sparse.coo_matrix((di, (rowi, coli)), shape=(N, N)).tocsr()
    
    return Dx, Dy

# ═══════════════════════════════════════════════════════════════════
# DIAGONAL PRECONDITIONER (for fast PCG solver)
# ═══════════════════════════════════════════════════════════════════

class DiagonalPreconditioner(LinearOperator):
    """Diagonal preconditioner for conjugate gradient solver"""
    def __init__(self, A):
        self.shape = A.shape
        self.dtype = A.dtype
        self.diag_inv = 1.0 / (A.diagonal() + 1e-8)
    
    def _matvec(self, x):
        return self.diag_inv * x

# ═══════════════════════════════════════════════════════════════════
# ENHANCED VNDHR VARIATIONAL MODEL (Algorithm 1 from paper)
# ═══════════════════════════════════════════════════════════════════

class EnhancedVNDHRVariational:
    """
    Implements the Hybrid Variational Model (HVM) from Section III-B
    Decomposes image into illumination and reflectance using:
    - ℓp norm for structure-aware illumination
    - Weighted ℓ2 norm for fine structures in reflectance  
    - ℓ1 TV norm for noise suppression
    """
    
    def __init__(self, img_np, params=None):
        self.S = img_np.astype(np.float32)
        self.H, self.W = self.S.shape[:2]
        
        # Load parameters
        if params is None:
            params = VNDHR_PARAMS
        self.p = params.get('p', 0.65)
        self.lambda1 = params.get('lambda1', 0.002)
        self.lambda2 = params.get('lambda2', 0.0001)
        self.lambda3 = params.get('lambda3', 0.001)
        self.max_iter = params.get('max_iter', 10)
        self.tau1 = params.get('tau1', 1e-6)
        self.tau2 = params.get('tau2', 1e-6)
        
        # Convert to HSV and extract V-channel (as per paper)
        hsv = cv2.cvtColor((self.S * 255).astype(np.uint8), 
                          cv2.COLOR_RGB2HSV).astype(np.float32) / 255.0
        self.I = hsv[:, :, 2].copy()  # Illumination (V-channel)
        self.R = np.ones_like(self.I)  # Reflectance initialization
        
        # Construct gradient operators
        self.Dx, self.Dy = gradient_operators(self.H, self.W)
    
    def compute_GI(self, I):
        """Compute weight matrix GI for ℓp norm (Eq. 8)"""
        gradIx = self.Dx.dot(I.flatten()).reshape(self.H, self.W)
        gradIy = self.Dy.dot(I.flatten()).reshape(self.H, self.W)
        magI = np.sqrt(gradIx**2 + gradIy**2)
        return np.power(np.maximum(magI, self.tau1), self.p - 2)
    
    def compute_WR(self, R):
        """Compute weight matrix WR for weighted ℓ2 norm (Eq. 6)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return 1.0 / (np.maximum(magR, self.tau2) + 1e-8)
    
    def compute_GR(self, R):
        """Compute weight matrix GR for ℓ1 norm (Eq. 9)"""
        gradRx = self.Dx.dot(R.flatten()).reshape(self.H, self.W)
        gradRy = self.Dy.dot(R.flatten()).reshape(self.H, self.W)
        magR = np.sqrt(gradRx**2 + gradRy**2)
        return np.maximum(magR, self.tau2) ** (-1)
    
    def solve_I_subproblem(self, R, GI):
        """Solve I sub-problem using PCG (Eq. 13)"""
        Rdiag = sparse.diags(R.flatten())
        Wgi = sparse.diags(GI.flatten())
        U = self.Dx.T.dot(Wgi.dot(self.Dx)) + self.Dy.T.dot(Wgi.dot(self.Dy))
        
        A_I = Rdiag.T.dot(Rdiag) + self.lambda1 * U
        b_I = Rdiag.T.dot(self.I.flatten())
        
        M = DiagonalPreconditioner(A_I)
        x0 = self.I.flatten()
        
        x_sol, info = cg(A_I, b_I, x0=x0, M=M, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            x_sol = x0
        
        return x_sol.reshape(self.H, self.W)
    
    def solve_R_subproblem(self, I, WR, GR):
        """Solve R sub-problem using PCG (Eq. 17)"""
        Idiag = sparse.diags(I.flatten())
        Wwr = sparse.diags(WR.flatten())
        Wgr = sparse.diags(GR.flatten())
        
        V = self.Dx.T.dot(Wwr.dot(self.Dx)) + self.Dy.T.dot(Wwr.dot(self.Dy))
        M_term = self.Dx.T.dot(Wgr.dot(self.Dx)) + self.Dy.T.dot(Wgr.dot(self.Dy))
        
        LHS_R = Idiag.T.dot(Idiag) + self.lambda2 * V + self.lambda3 * M_term
        b_R = Idiag.T.dot(I.flatten())
        
        M_prec = DiagonalPreconditioner(LHS_R)
        r0 = self.R.flatten()
        
        r_sol, info = cg(LHS_R, b_R, x0=r0, M=M_prec, atol=1e-5, rtol=1e-5, maxiter=300)
        if info != 0:
            r_sol = r0
        
        return np.clip(r_sol.reshape(self.H, self.W), 0, 2.5)
    
    def run(self):
        """Run iterative optimization (Algorithm 1)"""
        for iteration in range(self.max_iter):
            # Compute weight matrices
            GI = self.compute_GI(self.I)
            WR = self.compute_WR(self.R)
            GR = self.compute_GR(self.R)
            
            # Update I and R
            I_new = self.solve_I_subproblem(self.R, GI)
            R_new = self.solve_R_subproblem(I_new, WR, GR)
            
            # Check convergence
            if (np.linalg.norm(I_new - self.I) / np.linalg.norm(self.I) < 1e-3 and
                np.linalg.norm(R_new - self.R) / np.linalg.norm(self.R) < 1e-3):
                break
            
            self.I = I_new
            self.R = R_new
        
        return self.I, self.R

# ═══════════════════════════════════════════════════════════════════
# PREPROCESSING FUNCTION
# ═══════════════════════════════════════════════════════════════════

def process_vndhr_folder(input_dir, output_dir, img_size=IMG_SIZE):
    """
    Process all images in a folder using VNDHR
    
    Args:
        input_dir: Directory containing input hazy images
        output_dir: Directory to save VNDHR preprocessed results
        img_size: Resize dimension for processing
    """
    files = sorted(glob.glob(os.path.join(input_dir, "*.*")))
    
    if len(files) == 0:
        print(f"⚠️  No images found in {input_dir}")
        return 0
    
    print(f"Processing {len(files)} images from {os.path.basename(input_dir)}")
    success_count = 0
    
    for path in tqdm(files, desc=f"VNDHR → {os.path.basename(output_dir)}"):
        try:
            # Load and resize image
            img = Image.open(path).convert("RGB")
            img = img.resize((img_size, img_size), Image.LANCZOS)
            img_np = np.array(img) / 255.0
            
            # Apply VNDHR decomposition
            vndhr = EnhancedVNDHRVariational(img_np, params=VNDHR_PARAMS)
            I, R = vndhr.run()
            
            # Reconstruct VNDHR output: S' = I ◦ R
            out = np.clip(img_np * I[:, :, None] * R[:, :, None], 0, 1)
            
            # Save result
            base = os.path.basename(path)
            save_path = os.path.join(output_dir, base)
            Image.fromarray((out * 255).astype(np.uint8)).save(save_path, quality=95)
            
            success_count += 1
            
        except Exception as e:
            print(f"\n⚠️  Error processing {os.path.basename(path)}: {e}")
    
    print(f"✓ Successfully processed {success_count}/{len(files)} images\n")
    return success_count

# ═══════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ═══════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("\n" + "═"*80)
    print("STEP 1: Processing Training Data")
    print("═"*80)
    train_count = process_vndhr_folder(TRAIN_INPUT, TRAIN_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("STEP 2: Processing Testing Data")
    print("═"*80)
    test_count = process_vndhr_folder(TEST_INPUT, TEST_VNDHR, IMG_SIZE)
    
    print("═"*80)
    print("VNDHR PREPROCESSING COMPLETE")
    print("═"*80)
    print(f"Training images processed: {train_count}")
    print(f"Testing images processed:  {test_count}")
    print(f"\nOutput directories:")
    print(f"  Train: {TRAIN_VNDHR}")
    print(f"  Test:  {TEST_VNDHR}")
    print("═"*80 + "\n")

In [ ]:
# -----------------------------
# Unified VNDHR Training: Complete 100-Epoch Pipeline
# Combines perceptual loss, fine-tuning, and aggressive augmentation
# Target: PSNR 28+ dB, SSIM 0.8+
# -----------------------------
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchvision.transforms.functional as TF
import os, glob
from PIL import Image, ImageEnhance
import torch.nn as nn
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from skimage.metrics import structural_similarity as compare_ssim
import numpy as np

# --- Paths ---
TRAIN_VNDHR = r"D:\Downloads\data\train_vndhr"
TRAIN_TARGET = r"D:\Downloads\data\train\target"
MODEL_CHECKPOINT = r"D:\Downloads\data\unet_vndhr_fixed.pth"  # Starting checkpoint
MODEL_SAVE = r"D:\Downloads\data\unet_vndhr_aggressive.pth"

# --- Hyperparameters ---
IMG_SIZE = 256
BATCH_SIZE = 6
TOTAL_EPOCHS = 100
NUM_WORKERS = 0
ENABLE_AMP = True

# Learning rate schedule
LR_STAGE1 = 3e-5   # Epochs 0-40: Perceptual loss introduction
LR_STAGE2 = 5e-6   # Epochs 41-70: Fine-tuning
LR_STAGE3 = 5e-5   # Epochs 71-100: Aggressive push
LR_MIN = 1e-7

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

# --- Perceptual Loss (VGG-based) ---
class VGGPerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(pretrained=True).features[:16].eval()
        for param in vgg.parameters():
            param.requires_grad = False
        self.vgg = vgg.to(device)
        self.mse = nn.MSELoss()
        
    def forward(self, pred, target):
        pred_feat = self.vgg(pred)
        target_feat = self.vgg(target)
        return self.mse(pred_feat, target_feat)

# --- Progressive Augmentation Dataset ---
class AdaptiveVNDHRDataset(Dataset):
    def __init__(self, vndhr_dir, clear_dir, augmentation_level='medium'):
        v_files = sorted(glob.glob(os.path.join(vndhr_dir, "*.*")))
        c_files = sorted(glob.glob(os.path.join(clear_dir, "*.*")))
        c_dict = {os.path.basename(f): f for f in c_files}
        self.pairs = [(v, c_dict[os.path.basename(v)]) for v in v_files if os.path.basename(v) in c_dict]
        self.aug_level = augmentation_level
        print(f"Dataset: {len(self.pairs)} pairs | Augmentation: {augmentation_level}")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        v_path, c_path = self.pairs[idx]
        v_img = Image.open(v_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)
        c_img = Image.open(c_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.LANCZOS)

        # Basic augmentation (all stages)
        if np.random.rand() > 0.5:
            v_img, c_img = TF.hflip(v_img), TF.hflip(c_img)
        if np.random.rand() > 0.5:
            v_img, c_img = TF.vflip(v_img), TF.vflip(c_img)
        if np.random.rand() > 0.5:
            angle = int(np.random.choice([90, 180, 270]))
            v_img, c_img = TF.rotate(v_img, angle), TF.rotate(c_img, angle)

        # Medium augmentation (stages 2+)
        if self.aug_level in ['medium', 'heavy']:
            if np.random.rand() > 0.7:
                brightness = np.random.uniform(0.95, 1.05)
                contrast = np.random.uniform(0.95, 1.05)
                v_img = ImageEnhance.Brightness(v_img).enhance(brightness)
                v_img = ImageEnhance.Contrast(v_img).enhance(contrast)
                c_img = ImageEnhance.Brightness(c_img).enhance(brightness)
                c_img = ImageEnhance.Contrast(c_img).enhance(contrast)

        # Heavy augmentation (stage 3 only)
        if self.aug_level == 'heavy':
            if np.random.rand() > 0.8:
                crop_size = int(IMG_SIZE * np.random.uniform(0.85, 0.95))
                i, j, h, w = transforms.RandomCrop.get_params(v_img, (crop_size, crop_size))
                v_img = TF.resized_crop(v_img, i, j, h, w, (IMG_SIZE, IMG_SIZE))
                c_img = TF.resized_crop(c_img, i, j, h, w, (IMG_SIZE, IMG_SIZE))

        return transforms.ToTensor()(v_img), transforms.ToTensor()(c_img)

# --- UNet Architecture ---
class ImprovedUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, base_c=64):
        super().__init__()
        self.enc1 = self._block(in_ch, base_c)
        self.enc2 = self._block(base_c, base_c*2)
        self.enc3 = self._block(base_c*2, base_c*4)
        self.enc4 = self._block(base_c*4, base_c*8)
        self.bottleneck = self._block(base_c*8, base_c*16)
        self.up4 = nn.ConvTranspose2d(base_c*16, base_c*8, 2, stride=2)
        self.dec4 = self._block(base_c*16, base_c*8)
        self.up3 = nn.ConvTranspose2d(base_c*8, base_c*4, 2, stride=2)
        self.dec3 = self._block(base_c*8, base_c*4)
        self.up2 = nn.ConvTranspose2d(base_c*4, base_c*2, 2, stride=2)
        self.dec2 = self._block(base_c*4, base_c*2)
        self.up1 = nn.ConvTranspose2d(base_c*2, base_c, 2, stride=2)
        self.dec1 = self._block(base_c*2, base_c)
        self.final_conv = nn.Conv2d(base_c, out_ch, 1)
        self.pool = nn.MaxPool2d(2)
        
    def _block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.final_conv(d1))

# --- Load Initial Model ---
print(f"{'='*70}")
print("UNIFIED 100-EPOCH TRAINING PIPELINE")
print(f"{'='*70}\n")

model = ImprovedUNet(base_c=64).to(device)
checkpoint = torch.load(MODEL_CHECKPOINT, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
best_psnr = checkpoint['psnr']
start_epoch = checkpoint['epoch']

print(f"✓ Loaded checkpoint from epoch {start_epoch}")
print(f"✓ Starting PSNR: {best_psnr:.2f} dB | Target: 28+ dB\n")

# --- Training Setup ---
mse_loss = nn.MSELoss()
l1_loss = nn.L1Loss()
perceptual_loss = VGGPerceptualLoss()
scaler = torch.amp.GradScaler(enabled=ENABLE_AMP)

patience = 0
max_patience = 20
current_stage = 1

print(f"{'='*70}")
print("TRAINING STAGES:")
print(f"{'='*70}")
print("Stage 1 (Epochs 0-40):   Perceptual Loss Introduction")
print("Stage 2 (Epochs 41-70):  Conservative Fine-tuning")
print("Stage 3 (Epochs 71-100): Aggressive Optimization")
print(f"{'='*70}\n")

# --- Main Training Loop ---
for epoch in range(TOTAL_EPOCHS):
    # Stage transitions
    if epoch == 0:
        current_stage = 1
        lr = LR_STAGE1
        aug_level = 'light'
        print(f"\n{'='*70}")
        print(f"STAGE 1: Perceptual Loss Introduction (Epochs 0-40)")
        print(f"{'='*70}")
        print(f"LR: {lr:.2e} | Augmentation: {aug_level}")
        dataset = AdaptiveVNDHRDataset(TRAIN_VNDHR, TRAIN_TARGET, aug_level)
        dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
        
    elif epoch == 41:
        current_stage = 2
        lr = LR_STAGE2
        aug_level = 'medium'
        print(f"\n{'='*70}")
        print(f"STAGE 2: Conservative Fine-tuning (Epochs 41-70)")
        print(f"{'='*70}")
        print(f"LR: {lr:.2e} | Augmentation: {aug_level}")
        dataset = AdaptiveVNDHRDataset(TRAIN_VNDHR, TRAIN_TARGET, aug_level)
        dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=8)
        
    elif epoch == 71:
        current_stage = 3
        lr = LR_STAGE3
        aug_level = 'heavy'
        print(f"\n{'='*70}")
        print(f"STAGE 3: Aggressive Optimization (Epochs 71-100)")
        print(f"{'='*70}")
        print(f"LR: {lr:.2e} | Augmentation: {aug_level}")
        dataset = AdaptiveVNDHRDataset(TRAIN_VNDHR, TRAIN_TARGET, aug_level)
        dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2, eta_min=LR_MIN
        )

    # Determine loss weights based on stage
    if current_stage == 1:
        # Gradually introduce perceptual loss
        if epoch < 20:
            w_pixel, w_perc = 1.0, 0.0
        elif epoch < 40:
            progress = (epoch - 20) / 20
            w_pixel, w_perc = 1.0, 0.1 * progress
        else:
            w_pixel, w_perc = 1.0, 0.1
    else:
        # Stages 2 & 3: Pure pixel loss for maximum PSNR
        w_pixel, w_perc = 1.0, 0.0

    # Training epoch
    model.train()
    epoch_loss = 0
    psnr_list, ssim_list = [], []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS} [Stage {current_stage}]")

    for v, c in pbar:
        v, c = v.to(device).float(), c.to(device).float()
        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda", enabled=ENABLE_AMP):
            pred = model(v)
            
            # Pixel losses
            loss_mse = mse_loss(pred, c)
            loss_l1 = l1_loss(pred, c)
            
            # Combined loss
            if w_perc > 0:
                with torch.cuda.amp.autocast(enabled=False):
                    loss_perc = perceptual_loss(pred.float(), c.float())
                loss = w_pixel * (0.5*loss_mse + 0.5*loss_l1) + w_perc * loss_perc
            else:
                loss = 0.6 * loss_mse + 0.4 * loss_l1

        if torch.isnan(loss) or torch.isinf(loss):
            continue

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

        # Metrics
        with torch.no_grad():
            pred_np = (pred.clamp(0,1).cpu().numpy()*255).astype(np.uint8)
            c_np = (c.cpu().numpy()*255).astype(np.uint8)
            for i in range(pred_np.shape[0]):
                psnr_list.append(compare_psnr(c_np[i].transpose(1,2,0), pred_np[i].transpose(1,2,0), data_range=255))
                ssim_list.append(compare_ssim(c_np[i].transpose(1,2,0), pred_np[i].transpose(1,2,0), channel_axis=2, data_range=255))

        pbar.set_postfix(loss=f"{loss.item():.4f}", psnr=f"{np.mean(psnr_list):.2f}", ssim=f"{np.mean(ssim_list):.3f}")

    # Epoch summary
    avg_psnr = np.mean(psnr_list)
    avg_ssim = np.mean(ssim_list)
    avg_loss = epoch_loss / len(dataloader)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch+1}/{TOTAL_EPOCHS} | Stage {current_stage}")
    print(f"  Loss: {avg_loss:.4f} | PSNR: {avg_psnr:.2f} dB | SSIM: {avg_ssim:.4f}")
    print(f"  Weights - Pixel: {w_pixel:.2f}, Perceptual: {w_perc:.2f} | LR: {current_lr:.2e}")
    
    # Scheduler step
    if current_stage in [1, 2]:
        scheduler.step(avg_psnr)
    else:
        scheduler.step()

    # Save best model
    if avg_psnr > best_psnr:
        improvement = avg_psnr - best_psnr
        best_psnr = avg_psnr
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'psnr': avg_psnr,
            'ssim': avg_ssim,
            'stage': current_stage
        }, MODEL_SAVE)
        print(f"  ✓ NEW BEST! Improved by +{improvement:.2f} dB")
        patience = 0
    else:
        patience += 1
        print(f"  No improvement (patience: {patience}/{max_patience})")
    
    # Milestones
    if avg_psnr >= 26.0 and best_psnr < 26.0:
        print(f"  🎯 Milestone: 26+ dB reached!")
    if avg_psnr >= 28.0:
        print(f"  🎉 TARGET REACHED! PSNR: {avg_psnr:.2f} dB, SSIM: {avg_ssim:.4f}")
        break
        
    # Early stopping
    if patience >= max_patience:
        print(f"\n⚠️ Early stopping at epoch {epoch+1}")
        break

# --- Final Summary ---
print(f"\n{'='*70}")
print("TRAINING COMPLETE")
print(f"{'='*70}")
print(f"Starting PSNR:  {checkpoint['psnr']:.2f} dB")
print(f"Final PSNR:     {best_psnr:.2f} dB")
print(f"Improvement:    +{best_psnr - checkpoint['psnr']:.2f} dB")
print(f"\nExpected test performance: ~{best_psnr + 1.28:.2f} dB (with TTA)")
print(f"\nModel saved to: {MODEL_SAVE}")
print(f"{'='*70}")